# Préparation des données SFT — UltraMedical (anglais)

Partie **anglaise** du SFT : questions de [TsinghuaC3I/UltraMedical-Preference](https://huggingface.co/datasets/TsinghuaC3I/UltraMedical-Preference) (licence MIT), avec leur meilleure réponse (`chosen`) comme réponse attendue.

- **Pourquoi** : le DPO anglais (`data_prep_DPO_UltraMedical`) part du modèle SFT. Le SFT doit donc avoir appris l'anglais et le style des réponses UltraMedical (raisonnement, puis « So, the answer is C. » pour les QCM) avant que le DPO n'ajuste des préférences.
- **Pas de recouvrement avec le DPO** : aucune question des paires DPO n'est reprise. Un contrôle final le vérifie.
- **Qualité** : pour les QCM, la réponse doit donner la bonne lettre d'après le corrigé (`golden_answer`) ; pour toutes les sources, la note GPT-4 de la réponse doit être élevée.
- **Sortie** : `data/anonymized/sft_ultramedical_{train,validation,test}.anonymized.jsonl`, au format `messages` du SFT.

# Import

In [ ]:
import json
import re

import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer

from projet_14.rgpd import run

SEED = 42
MODEL_NAME = "Qwen/Qwen3-1.7B-Base"  # pour compter les tokens avec le tokenizer du modèle entraîné

# Sources QCM : la bonne réponse (golden_answer) est une lettre, vérifiable dans la réponse
MCQ_SOURCES = ["MedQA", "MedMCQA", "MedQA-Evol", "PubMedQA", "TextBookQA"]
MIN_CHOSEN_SCORE = 4.5  # note GPT-4 minimale de la réponse (de 1 à 5)

# Longueurs maximales en tokens : MAX_LENGTH est celui du SFT (SFTConfig.max_length)
MAX_PROMPT_TOKENS = 512
MAX_LENGTH = 1024
TEMPLATE_TOKENS = 40  # system prompt et balises du gabarit de conversation

# Taille maximale des jeux
N_TRAIN = 1500
N_VALIDATION = 300
N_TEST = 500

# Get hugging face dataset

In [ ]:
# Environ 1 Go à télécharger (fichier train.json)
ds = load_dataset("TsinghuaC3I/UltraMedical-Preference")
print(ds)

In [ ]:
def flatten(df: pd.DataFrame) -> pd.DataFrame:
    """Une ligne par paire : question, meilleure réponse et ses métadonnées."""
    meta = pd.json_normalize(df["metadata"].tolist())
    return pd.DataFrame(
        {
            "prompt_id": df["prompt_id"],
            "source": df["prompt_id"].str.split(",").str[0],  # "MedQA,1234" -> "MedQA"
            "prompt": df["prompt"].str.strip(),
            "answer": df["chosen"].map(lambda messages: messages[-1]["content"].strip()),
            "chosen_model": meta["chosen.model"],
            "chosen_score": meta["chosen.score"],
            "golden_answer": meta["golden_answer"],
        }
    )


raw = {name: flatten(ds[name].to_pandas()) for name in ds}

# Analyse dataset

In [ ]:
raw["train"].head()

In [ ]:
# Une même question apparaît dans plusieurs paires (easy, hard, length) : une seule ligne par question sera gardée
for name, df in raw.items():
    print(f"{name}: {len(df)} paires, {df['prompt'].nunique()} questions distinctes")

In [ ]:
raw["train"]["source"].value_counts()

# Questions exclues

- **Questions du DPO** : toutes celles des fichiers `dpo_ultramedical_*` (avant anonymisation).
- **Fuite entre splits** : une question du train présente aussi en validation ou en test d'UltraMedical.

In [ ]:
dpo_prompts = set()
for name in ["train", "validation", "test"]:
    with open(f"../data/dpo_ultramedical_{name}.jsonl", encoding="utf-8") as f:
        dpo_prompts |= {json.loads(line)["prompt"][-1]["content"] for line in f}
print(f"{len(dpo_prompts)} questions utilisées par le DPO")

held_out = set(raw["validation"]["prompt"]) | set(raw["test"]["prompt"])
for name in ["train", "validation"]:
    df = raw[name]
    n_before = len(df)
    df = df[~df["prompt"].isin(dpo_prompts)]
    if name == "train":
        df = df[~df["prompt"].isin(held_out)]
    raw[name] = df
    print(f"{name}: {n_before} paires, {len(df)} après exclusion")

# Qualité des réponses

- **QCM** : la lettre donnée par la réponse (« So, the answer is C. ») doit être celle du corrigé.
- **Toutes les sources** : note GPT-4 de la réponse ≥ `MIN_CHOSEN_SCORE`.
- **Une ligne par question** : la réponse la mieux notée.

In [ ]:
ANSWER = re.compile(r"answer is:?\s*\(?\**([A-J])\b", re.IGNORECASE)


def answer_letter(text: str) -> str | None:
    """Lettre donnée comme réponse finale, ou None si absente."""
    found = ANSWER.findall(text)
    return found[-1].upper() if found else None


for name in ["train", "validation"]:
    df = raw[name]
    is_mcq = df["source"].isin(MCQ_SOURCES)
    correct = df["answer"].map(answer_letter) == df["golden_answer"]
    good = (~is_mcq | correct) & (df["chosen_score"] >= MIN_CHOSEN_SCORE) & (df["answer"] != "")
    n_before = len(df)
    df = df[good].assign(pair_check=lambda d: d["source"].isin(MCQ_SOURCES).map({True: "golden_answer", False: "gpt4_score"}))
    raw[name] = df.sort_values("chosen_score", ascending=False, kind="stable").drop_duplicates(subset="prompt")
    print(f"{name}: {n_before} paires, {len(df)} réponses de qualité, {len(raw[name])} questions distinctes")

# Filtre sur la longueur (tokens)

Une réponse coupée par `max_length` perdrait sa fin, et avec elle le token de fin `<|endoftext|>` : le modèle apprendrait à ne pas s'arrêter. On ne garde que les exemples qui tiennent entiers.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def n_tokens(texts: pd.Series) -> list[int]:
    return [len(ids) for ids in tokenizer(texts.tolist(), add_special_tokens=False)["input_ids"]]


for name in ["train", "validation"]:
    df = raw[name]
    df = df.assign(n_prompt=n_tokens(df["prompt"]), n_answer=n_tokens(df["answer"]))
    n_before = len(df)
    df = df[(df["n_prompt"] <= MAX_PROMPT_TOKENS) & (df["n_prompt"] + df["n_answer"] + TEMPLATE_TOKENS <= MAX_LENGTH)]
    raw[name] = df
    print(f"{name}: {n_before} rows, {len(df)} après filtre sur la longueur")

raw["train"][["n_prompt", "n_answer"]].describe().round(0)

# Découpage train / validation / test

Le test d'UltraMedical est entièrement utilisé par le DPO. La validation et le test du SFT sont donc tirés, sans recouvrement, de la validation d'UltraMedical.

In [ ]:
validation_pool = raw["validation"].sample(frac=1, random_state=SEED)
splits = {
    "train": raw["train"].sample(n=min(N_TRAIN, len(raw["train"])), random_state=SEED),
    "validation": validation_pool.iloc[:N_VALIDATION],
    "test": validation_pool.iloc[N_VALIDATION : N_VALIDATION + N_TEST],
}
for name, df in splits.items():
    print(f"{name}: {len(df)} rows")

In [ ]:
# Répartition par source
pd.concat({name: df["source"].value_counts() for name, df in splits.items()}, axis=1).fillna(0).astype(int)

# Build SFT JSONL file from dataset

In [ ]:
# Même system prompt que le DPO anglais
SYSTEM_PROMPT = "You are an expert medical assistant. Answer accurately and clearly."


def to_messages(row) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["answer"]},
    ]


METADATA = ["prompt_id", "source", "pair_check", "golden_answer", "chosen_model", "chosen_score", "lang"]

# Un fichier JSONL par split : ../data/sft_ultramedical_{train,validation,test}.jsonl
sft_ds = DatasetDict()
for name, df in splits.items():
    df = df.assign(lang="en", messages=df.apply(to_messages, axis=1))
    sft_ds[name] = Dataset.from_pandas(df[METADATA + ["messages"]], preserve_index=False)
    sft_ds[name].to_json(f"../data/sft_ultramedical_{name}.jsonl", force_ascii=False)

sft_ds["train"][0]["messages"]

In [ ]:
print(sft_ds)

# Anonymisation des données

Comme pour le DPO anglais, seule la source **`ChatDoctor`** contient de vrais échanges patients / médecins. On l'anonymise en entier (question et réponse : les réponses reprennent souvent le prénom du patient, « Hello Shonda »). Les autres sources (questions d'examen, manuels, Wikipédia, réponses générées) n'ont pas de donnée personnelle réelle.

In [ ]:
# Anonymisation de chaque split : ../data/anonymized/sft_ultramedical_{split}.anonymized.jsonl
reports = {
    name: run(f"../data/sft_ultramedical_{name}.jsonl", lang="en", output_dir="../data/anonymized", sources=["ChatDoctor"])
    for name in sft_ds
}
pd.DataFrame(reports).T[["records", "records_scanned", "records_modified", "persons_replaced"]]

In [ ]:
# Contexte de chaque balise <PERSON>, pour vérification
rows = []
for name in sft_ds:
    with open(f"../data/anonymized/sft_ultramedical_{name}.anonymized.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            for message in record["messages"]:
                if message["role"] != "system":
                    rows.append({"split": name, "source": record["source"], "role": message["role"], "content": message["content"]})
contents_df = pd.DataFrame(rows)


def person_contexts(text: str, width: int = 60) -> list[str]:
    """Extrait autour de chaque balise <PERSON>, avec `width` caractères de chaque côté."""
    return [text[max(0, m.start() - width) : m.end() + width].replace("\n", " ") for m in re.finditer("<PERSON>", text)]


occurrences_df = contents_df[["split", "source", "role"]].copy()
occurrences_df["context"] = contents_df["content"].apply(person_contexts)
occurrences_df = occurrences_df.explode("context").dropna(subset=["context"])

pd.set_option("display.max_colwidth", None)
print(f"{len(occurrences_df)} occurrences de <PERSON>")
occurrences_df.sample(min(20, len(occurrences_df)), random_state=SEED)

# Contrôle final

In [ ]:
# Rechargement des fichiers anonymisés, tels que les lira le notebook d'entraînement
DATA_FILES = {name: f"../data/anonymized/sft_ultramedical_{name}.anonymized.jsonl" for name in sft_ds}
final_ds = load_dataset("json", data_files=DATA_FILES)
print(final_ds)

# Aucune question commune avec le DPO, ni entre les splits du SFT
sft_prompts = {name: set(df["prompt"]) for name, df in splits.items()}
assert not set.union(*sft_prompts.values()) & dpo_prompts, "questions communes avec le DPO"
assert not sft_prompts["train"] & (sft_prompts["validation"] | sft_prompts["test"]), "fuite train / évaluation"
assert not sft_prompts["validation"] & sft_prompts["test"], "fuite validation / test"
print("OK : aucune question commune avec le DPO ni entre les splits")

# Limites

- **Réponses générées par des LLM** (GPT-4, Llama-3…), jugées par GPT-4. Pour les QCM, la bonne réponse est vérifiée par le corrigé ; le raisonnement qui y mène ne l'est pas.
- **Réponses longues** (souvent plusieurs centaines de tokens), alors que le SFT français apprend des réponses courtes : le modèle apprend deux styles, selon la langue et le type de question.